# Latent Baseline Benchmark

Ноутбук для classifier-only сравнения `IG`, `NAA` и grid-конфигураций `Cheap-IG` на `100` изображениях из `Oxford Pets` в latent-space benchmark без `ROAD`-импутации. Вместо `ROAD` используются baseline donors / latent imputers, а primary score — `target logit drop AOC20` по top-`1..20%` нейронов слоя.

По умолчанию baseline-порядок начинается с `black_act`.

In [1]:
from pathlib import Path

from IPython.display import Markdown, display

from modules.latent_baseline_benchmark import (
    benchmark_classifier_latent_baseline,
    classifier_method_spec,
    render_latent_baseline_report,
)


In [2]:
OXFORD_PETS_DIR = Path("oxford_pets")
N_IMAGES = 100

CLASSIFIER_LAYER = "model.6"
N_STEPS = 128
BUDGET_PERCENTILES = list(range(1, 21))
DONOR_KINDS = [
    "black_act",
    "zero_baseline",
    "blur_act",
    "layer_mean_exclusive",
    "spatial_nli_same_channel",
]
BLUR_SIGMA = 16.0
PREVIEW_IMAGES = 5
CLEAR_EVERY = 8
FD_EPS = 1e-3

CHEAP_IG_SEGMENT_START = 0.0
CHEAP_IG_SEGMENT_END = 0.2
CHEAP_IG_SELECTION_MODE = "positive"
CHEAP_IG_SELECTION_TOP_K_VALUES = [8000, 16000, 32000]

CACHE_ROOT = Path("output/latent_baseline_cache")
OUTPUT_DIR = Path("output/latent_baseline_classifier_oxford_pets_100")
REFRESH_CORE = False
REFRESH_METHODS = False
REFRESH_EVALUATIONS = False


In [3]:
def collect_image_paths(root: Path, n_images: int):
    exts = {".jpg", ".jpeg", ".png", ".webp"}
    paths = sorted(
        [path for path in root.iterdir() if path.suffix.lower() in exts],
        key=lambda path: path.name.lower(),
    )
    return [str(path) for path in paths[:n_images]]


IMAGE_PATHS = collect_image_paths(OXFORD_PETS_DIR, N_IMAGES)
len(IMAGE_PATHS), IMAGE_PATHS[:3]


(100,
 ['oxford_pets/Abyssinian_1.jpg',
  'oxford_pets/Abyssinian_108.jpg',
  'oxford_pets/Abyssinian_117.jpg'])

In [4]:
def build_cheap_ig_variants(fill_mode, fill_rho=None):
    variants = []
    for top_k in CHEAP_IG_SELECTION_TOP_K_VALUES:
        suffix = fill_mode if fill_mode == "zero" else f"{fill_mode}/rho{fill_rho:g}"
        variants.append(
            classifier_method_spec(
                "cheap_ig",
                name=f"Cheap-IG+[0,0.2]/k{top_k}/{suffix}",
                segment_start=CHEAP_IG_SEGMENT_START,
                segment_end=CHEAP_IG_SEGMENT_END,
                selection_mode=CHEAP_IG_SELECTION_MODE,
                selection_top_k=top_k,
                fill_mode=fill_mode,
                fill_rho=fill_rho if fill_rho is not None else 0.8,
            )
        )
    return variants


METHOD_SPECS = [
    classifier_method_spec("ig", name="IG"),
    classifier_method_spec("naa", name="NAA"),
    *build_cheap_ig_variants("zero"),
    *build_cheap_ig_variants("naa_scaled", fill_rho=0.8),
    *build_cheap_ig_variants("naa_scaled", fill_rho=1.0),
]

len(METHOD_SPECS), [spec["name"] for spec in METHOD_SPECS]


(11,
 ['IG',
  'NAA',
  'Cheap-IG+[0,0.2]/k8000/zero',
  'Cheap-IG+[0,0.2]/k16000/zero',
  'Cheap-IG+[0,0.2]/k32000/zero',
  'Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8',
  'Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8',
  'Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8',
  'Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1',
  'Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1',
  'Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1'])

In [5]:
results = benchmark_classifier_latent_baseline(
    image_paths=IMAGE_PATHS,
    method_specs=METHOD_SPECS,
    layer_name=CLASSIFIER_LAYER,
    n_steps=N_STEPS,
    budget_percentiles=BUDGET_PERCENTILES,
    donor_kinds=DONOR_KINDS,
    blur_sigma=BLUR_SIGMA,
    preview_images=PREVIEW_IMAGES,
    clear_every=CLEAR_EVERY,
    fd_eps=FD_EPS,
    cache_root=CACHE_ROOT,
    target_dir=OUTPUT_DIR,
    save_output=True,
    refresh_core=REFRESH_CORE,
    refresh_methods=REFRESH_METHODS,
    refresh_evaluations=REFRESH_EVALUATIONS,
    verbose=False,
)

print("output_dir:", results["output_dir"])
print("report_path:", results["report_path"])
print("summary_path:", results["summary_path"])


output_dir: /Users/ashentide/PycharmProjects/PaperImplementations/output/latent_baseline_classifier_oxford_pets_100
report_path: /Users/ashentide/PycharmProjects/PaperImplementations/output/latent_baseline_classifier_oxford_pets_100/latent_baseline_report.md
summary_path: /Users/ashentide/PycharmProjects/PaperImplementations/output/latent_baseline_classifier_oxford_pets_100/latent_baseline_summary.json


In [6]:
artifacts = render_latent_baseline_report(results, output_dir=OUTPUT_DIR)
artifacts["report_path"], artifacts["summary_path"]


('output/latent_baseline_classifier_oxford_pets_100/latent_baseline_report.md',
 'output/latent_baseline_classifier_oxford_pets_100/latent_baseline_summary.json')

In [ ]:
display(Markdown(Path(artifacts["report_path"]).read_text(encoding="utf-8")))


# Latent Baseline Benchmark

Classifier-only benchmark for `yolo11s-cls` using latent deletion AOC20 with baseline donors instead of ROAD imputation.

## Configuration

- layer_name=`model.6`
- n_steps=`128`
- budget_percentiles=`[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]`
- donor_kinds=`['black_act', 'zero_baseline', 'blur_act', 'layer_mean_exclusive', 'spatial_nli_same_channel']`
- blur_sigma=`16.0`
- n_images=`100`

## Aggregate Heatmaps

![](output/latent_baseline_classifier_oxford_pets_100/figures/latent_baseline_heatmap_aoc20.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/latent_baseline_heatmap_aoc20_norm.png)

## Summary Table

| Donor | Method | AOC20 mean | AOC20 std | AOC20 norm mean | runtime_s mean | benchmark_runtime_s mean | abs_error mean |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| zero_baseline | NAA | 15.0242 | 3.1167 | 1.0587 | 2.9431 | 3.2720 | 13.1379 |
| zero_baseline | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 13.5458 | 3.0956 | 0.9542 | 2.8148 | 3.1630 | 94.6795 |
| zero_baseline | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 13.5374 | 3.0956 | 0.9536 | 2.7718 | 3.1071 | 95.3462 |
| zero_baseline | Cheap-IG+[0,0.2]/k8000/zero | 13.3519 | 3.1042 | 0.9404 | 2.7087 | 3.0451 | 92.0125 |
| zero_baseline | Cheap-IG+[0,0.2]/k16000/zero | 13.2914 | 3.0880 | 0.9362 | 2.8053 | 3.1506 | 96.4712 |
| zero_baseline | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 13.2884 | 3.0891 | 0.9360 | 2.9244 | 3.2736 | 97.1905 |
| zero_baseline | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 13.2863 | 3.0894 | 0.9358 | 2.7517 | 3.0841 | 97.3697 |
| zero_baseline | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 13.1346 | 3.0947 | 0.9251 | 2.8454 | 3.1860 | 97.9022 |
| zero_baseline | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 13.1346 | 3.0947 | 0.9251 | 2.7272 | 3.0564 | 97.9022 |
| zero_baseline | Cheap-IG+[0,0.2]/k32000/zero | 13.1346 | 3.0947 | 0.9251 | 2.7979 | 3.1384 | 97.9022 |
| zero_baseline | IG | 13.0741 | 3.5192 | 0.9184 | 5.6534 | 5.9857 | 0.7237 |
| black_act | NAA | 15.9521 | 3.1673 | 1.1244 | 2.9431 | 3.2735 | 13.1379 |
| black_act | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 14.8108 | 3.1806 | 1.0435 | 2.8148 | 3.1604 | 94.6795 |
| black_act | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 14.8041 | 3.1807 | 1.0430 | 2.7718 | 3.1085 | 95.3462 |
| black_act | Cheap-IG+[0,0.2]/k8000/zero | 14.6599 | 3.2053 | 1.0325 | 2.7087 | 3.0426 | 92.0125 |
| black_act | Cheap-IG+[0,0.2]/k16000/zero | 14.5475 | 3.1882 | 1.0247 | 2.8053 | 3.1516 | 96.4712 |
| black_act | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 14.5468 | 3.1889 | 1.0246 | 2.9244 | 3.2825 | 97.1905 |
| black_act | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 14.5453 | 3.1892 | 1.0245 | 2.7517 | 3.0886 | 97.3697 |
| black_act | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 14.3801 | 3.1950 | 1.0128 | 2.8454 | 3.1855 | 97.9022 |
| black_act | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 14.3801 | 3.1950 | 1.0128 | 2.7272 | 3.0569 | 97.9022 |
| black_act | Cheap-IG+[0,0.2]/k32000/zero | 14.3801 | 3.1950 | 1.0128 | 2.7979 | 3.1440 | 97.9022 |
| black_act | IG | 14.2383 | 3.5475 | 1.0000 | 5.6534 | 5.9866 | 0.7237 |
| blur_act | NAA | 14.9987 | 3.1798 | 1.0561 | 2.9431 | 3.2718 | 13.1379 |
| blur_act | IG | 13.8002 | 3.4696 | 0.9697 | 5.6534 | 5.9869 | 0.7237 |
| blur_act | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 13.7395 | 3.1893 | 0.9673 | 2.8148 | 3.1666 | 94.6795 |
| blur_act | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 13.7332 | 3.1898 | 0.9668 | 2.7718 | 3.1120 | 95.3462 |
| blur_act | Cheap-IG+[0,0.2]/k8000/zero | 13.5828 | 3.2120 | 0.9561 | 2.7087 | 3.0425 | 92.0125 |
| blur_act | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 13.5698 | 3.2092 | 0.9552 | 2.9244 | 3.2753 | 97.1905 |
| blur_act | Cheap-IG+[0,0.2]/k16000/zero | 13.5690 | 3.2078 | 0.9552 | 2.8053 | 3.1447 | 96.4712 |
| blur_act | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 13.5683 | 3.2097 | 0.9551 | 2.7517 | 3.0831 | 97.3697 |
| blur_act | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 13.4676 | 3.2265 | 0.9478 | 2.8454 | 3.1875 | 97.9022 |
| blur_act | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 13.4676 | 3.2265 | 0.9478 | 2.7272 | 3.0572 | 97.9022 |
| blur_act | Cheap-IG+[0,0.2]/k32000/zero | 13.4676 | 3.2265 | 0.9478 | 2.7979 | 3.1375 | 97.9022 |
| layer_mean_exclusive | NAA | 15.2891 | 3.1583 | 1.0772 | 2.9431 | 3.3214 | 13.1379 |
| layer_mean_exclusive | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 13.8425 | 3.1312 | 0.9751 | 2.8148 | 3.2094 | 94.6795 |
| layer_mean_exclusive | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 13.8341 | 3.1313 | 0.9745 | 2.7718 | 3.1538 | 95.3462 |
| layer_mean_exclusive | Cheap-IG+[0,0.2]/k8000/zero | 13.6748 | 3.1450 | 0.9631 | 2.7087 | 3.0953 | 92.0125 |
| layer_mean_exclusive | Cheap-IG+[0,0.2]/k16000/zero | 13.5880 | 3.1284 | 0.9571 | 2.8053 | 3.1977 | 96.4712 |
| layer_mean_exclusive | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 13.5851 | 3.1293 | 0.9569 | 2.9244 | 3.3255 | 97.1905 |
| layer_mean_exclusive | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 13.5832 | 3.1296 | 0.9567 | 2.7517 | 3.1296 | 97.3697 |
| layer_mean_exclusive | IG | 13.4273 | 3.5081 | 0.9436 | 5.6534 | 6.0462 | 0.7237 |
| layer_mean_exclusive | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 13.4264 | 3.1362 | 0.9456 | 2.8454 | 3.2347 | 97.9022 |
| layer_mean_exclusive | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 13.4264 | 3.1362 | 0.9456 | 2.7272 | 3.1031 | 97.9022 |
| layer_mean_exclusive | Cheap-IG+[0,0.2]/k32000/zero | 13.4264 | 3.1362 | 0.9456 | 2.7979 | 3.1890 | 97.9022 |
| spatial_nli_same_channel | NAA | 15.0852 | 3.0044 | 1.0637 | 2.9431 | 3.8919 | 13.1379 |
| spatial_nli_same_channel | IG | 13.1452 | 3.3190 | 0.9252 | 5.6534 | 6.6213 | 0.7237 |
| spatial_nli_same_channel | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 12.9084 | 2.8487 | 0.9115 | 2.8148 | 3.8285 | 94.6795 |
| spatial_nli_same_channel | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 12.8946 | 2.8477 | 0.9105 | 2.7718 | 3.7316 | 95.3462 |
| spatial_nli_same_channel | Cheap-IG+[0,0.2]/k16000/zero | 12.5656 | 2.8381 | 0.8873 | 2.8053 | 3.7795 | 96.4712 |
| spatial_nli_same_channel | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 12.5617 | 2.8384 | 0.8870 | 2.9244 | 3.9320 | 97.1905 |
| spatial_nli_same_channel | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 12.5583 | 2.8384 | 0.8868 | 2.7517 | 3.6965 | 97.3697 |
| spatial_nli_same_channel | Cheap-IG+[0,0.2]/k8000/zero | 12.5007 | 2.8422 | 0.8824 | 2.7087 | 3.6973 | 92.0125 |
| spatial_nli_same_channel | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 12.3279 | 2.8389 | 0.8705 | 2.8454 | 3.8169 | 97.9022 |
| spatial_nli_same_channel | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 12.3279 | 2.8389 | 0.8705 | 2.7272 | 3.6730 | 97.9022 |
| spatial_nli_same_channel | Cheap-IG+[0,0.2]/k32000/zero | 12.3279 | 2.8389 | 0.8705 | 2.7979 | 3.7782 | 97.9022 |

## black_act

- best method by mean AOC20: `NAA`

![](output/latent_baseline_classifier_oxford_pets_100/figures/summary_black_act.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/distribution_black_act.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/curves_black_act.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/pairwise_black_act.png)

## zero_baseline

- best method by mean AOC20: `NAA`

![](output/latent_baseline_classifier_oxford_pets_100/figures/summary_zero_baseline.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/distribution_zero_baseline.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/curves_zero_baseline.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/pairwise_zero_baseline.png)

## blur_act

- best method by mean AOC20: `NAA`

![](output/latent_baseline_classifier_oxford_pets_100/figures/summary_blur_act.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/distribution_blur_act.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/curves_blur_act.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/pairwise_blur_act.png)

## layer_mean_exclusive

- best method by mean AOC20: `NAA`

![](output/latent_baseline_classifier_oxford_pets_100/figures/summary_layer_mean_exclusive.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/distribution_layer_mean_exclusive.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/curves_layer_mean_exclusive.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/pairwise_layer_mean_exclusive.png)

## spatial_nli_same_channel

- best method by mean AOC20: `NAA`

![](output/latent_baseline_classifier_oxford_pets_100/figures/summary_spatial_nli_same_channel.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/distribution_spatial_nli_same_channel.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/curves_spatial_nli_same_channel.png)

![](output/latent_baseline_classifier_oxford_pets_100/figures/pairwise_spatial_nli_same_channel.png)

## Visual Preview

Page 1 | methods: `['IG', 'NAA', 'Cheap-IG+[0,0.2]/k8000/zero', 'Cheap-IG+[0,0.2]/k16000/zero']`

![](output/latent_baseline_classifier_oxford_pets_100/figures/preview_page_1.png)

Page 2 | methods: `['Cheap-IG+[0,0.2]/k32000/zero', 'Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8', 'Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8', 'Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8']`

![](output/latent_baseline_classifier_oxford_pets_100/figures/preview_page_2.png)

Page 3 | methods: `['Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1', 'Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1', 'Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1']`

![](output/latent_baseline_classifier_oxford_pets_100/figures/preview_page_3.png)



: 